# Pendulum measurement: correlated uncertainty propagation

Estimate gravitational acceleration from a pendulum length $L$ and period
$P$ using the small-angle, point-mass model $g=4\pi^2L/P^2$. We neglect
finite-amplitude corrections, air drag, support motion, and mass distribution.
The input estimates and their joint normal distribution are synthetic.
Run the setup in [examples/README.md](README.md) first.

For standard uncertainties $u_L,u_P$ and correlation coefficient $r$, the
first-order propagation formula becomes
$$
\left(\frac{u_g}{g}\right)^2=
\left(\frac{u_L}{L}\right)^2+4\left(\frac{u_P}{P}\right)^2
-4r\frac{u_L}{L}\frac{u_P}{P}.
$$
The minus sign follows from the opposite signs of the sensitivities to
length and period. Correlation is an assumption about the input uncertainty
model; it is not inferred from the pendulum equation.

Reference: [JCGM 100, section 5.2: correlated input quantities](https://www.iso.org/sites/JCGM/GUM/JCGM100/C045315e-html/C045315e_FILES/MAIN_C045315e/05_e.html).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from unitx import Quantity
from unitx.si import si

length0 = 0.8 * si.m
period0 = 1.795 * si.s
length_uncertainty = 1.0 * si.mm
period_uncertainty = 3.0 * si.ms
correlation = 0.35
rng = np.random.default_rng(20261020)
standardized = rng.standard_normal((100_000, 2))
standardized[:, 1] = (
    correlation * standardized[:, 0] + np.sqrt(1 - correlation**2) * standardized[:, 1]
)
length = length0 + length_uncertainty * standardized[:, 0]
period = period0 + period_uncertainty * standardized[:, 1]
assert np.all(length.value > 0) and np.all(period.value > 0)
gravity = 4 * np.pi**2 * length / period**2

## Compare Monte Carlo moments with the local formula

NumPy's mean and standard deviation preserve the gravity unit; variance
has its square. These draws propagate uncertainty in the input estimates.
Increasing their count reduces simulation noise, not physical uncertainty,
so do not divide the propagated standard deviation by the square root of
the number of draws.

In [ ]:
central_gravity = 4 * np.pi**2 * length0 / period0**2
relative_length = length_uncertainty / length0
relative_period = period_uncertainty / period0
linear_uncertainty = (
    central_gravity
    * np.sqrt(
        relative_length**2
        + 4 * relative_period**2
        - 4 * correlation * relative_length * relative_period
    )
).to(gravity.unit)
simulated_mean = np.mean(gravity)
simulated_uncertainty = np.std(gravity, ddof=1)
simulated_variance = np.var(gravity, ddof=1)
assert isinstance(simulated_mean, Quantity)
assert isinstance(simulated_uncertainty, Quantity)
assert isinstance(simulated_variance, Quantity)

# Quantile is a numerical boundary; the existing unit is reattached directly.
interval = np.quantile(np.asarray(gravity.value), [0.025, 0.975]) * gravity.unit
print('Monte Carlo mean:', simulated_mean)
print('Propagated standard uncertainty:', simulated_uncertainty)
print('First-order standard uncertainty:', linear_uncertainty)
print('Central 95% interval under the input model:', interval)

## Inspect the input dependence and the output distribution

The Gaussian curve is the first-order approximation. The histogram and
quantiles describe the specified input model, not measured laboratory data
or a guarantee of frequentist coverage for unmodelled effects.

In [ ]:
g_axis = central_gravity + np.linspace(-4, 4, 400) * linear_uncertainty
normal_density = np.exp(-0.5 * ((g_axis - central_gravity) / linear_uncertainty) ** 2)
normal_density = normal_density / (np.sqrt(2 * np.pi) * linear_uncertainty)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), layout='constrained')
axes[0].scatter(length[:1500].value, period[:1500].value, s=6, alpha=0.2)
axes[0].set(
    xlabel=f'Length [{length.unit}]',
    ylabel=f'Period [{period.unit}]',
    title='Correlated input draws',
)
axes[1].hist(gravity.value, bins=70, density=True, alpha=0.6, label='Monte Carlo')
axes[1].plot(g_axis.value, normal_density.value, label='First-order Gaussian')
axes[1].set(
    xlabel=f'Gravity [{gravity.unit}]',
    ylabel=f'Probability density [{normal_density.unit}]',
    title='Propagated input uncertainty',
)
axes[1].legend()
plt.show()

## Check units and the small-uncertainty regime

A fixed random seed makes the example reproducible. The tolerance on the
Monte Carlo standard deviation allows for simulation fluctuations and the
small nonlinear correction to the first-order approximation.

In [ ]:
assert simulated_variance.unit == gravity.unit**2
assert simulated_variance.isclose(simulated_uncertainty**2, rel_tol=1e-12)
assert simulated_uncertainty.isclose(linear_uncertainty, rel_tol=0.02)
assert interval[0] < central_gravity < interval[1]
print(
    'Monte Carlo / first-order uncertainty:',
    simulated_uncertainty / linear_uncertainty,
)